# B8 · RL moderne : DQN, gradient de politique, PPO et RLHF — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b8_rl_moderne/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B8.Q1 à B8.Q12, B8.R1 à B8.R3, B8.1 à B8.6 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B8.12 à B8.15 | la valeur : CartPole pour un réseau, ton DQN avec sa mémoire de rejeu et son réseau cible, le DQN qui diverge, les ablations sur trois graines (*prochaine session de génération*) | | | |
| B | B8.16 à B8.24 | la politique : REINFORCE avec et sans baseline, GAE et l'objectif clippé dans ta librairie, ton PPO sur des environnements vectorisés, ses journaux, ses hyperparamètres et sa robustesse (*prochaine session de génération*) | | | |
| C | B8.25 à B8.28 | les préférences : Bradley-Terry et DPO dans ta librairie, un RLHF miniature sur CartPole, la revue de code face à CleanRL, et le défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Expliquer pourquoi et comment un réseau remplace la table Q, et ce qui rend DQN stable.
- Dériver le gradient de politique et justifier la baseline, l'avantage et GAE.
- Implémenter en PyTorch un DQN, REINFORCE et PPO sur CartPole, avec les calculs clés testés dans `mylearn`.
- Diagnostiquer un entraînement RL à partir de ses journaux et comparer des variantes sur plusieurs graines.
- Expliquer RLHF, DPO et GRPO et entraîner un modèle de récompense sur des préférences.

**Rappel express.** Cible DQN $y = r + \gamma(1 - d)\max_{a'} Q_{\bar\theta}(s', a')$ ; Double DQN : $Q_{\bar\theta}(s', \arg\max_{a'} Q_\theta(s', a'))$. Huber : $\frac12 x^2$ si $|x| \le \delta$, $\delta(|x| - \frac{\delta}{2})$ sinon. $\nabla_{\mathbf z} \log \operatorname{softmax}(\mathbf z)_a = \mathbf e_a - \operatorname{softmax}(\mathbf z)$. $\delta_t = r_t + \gamma(1 - \text{terminated}_t)V(s_{t+1}) - V(s_t)$, $\hat A_t = \delta_t + \gamma\lambda(1 - \text{done}_t)\hat A_{t+1}$. PPO : $-\operatorname{mean}\min(\rho \hat A, \operatorname{clip}(\rho, 1 \pm \varepsilon)\hat A)$. Bradley-Terry : $-\log\sigma(r_w - r_l)$ ; DPO : $-\log\sigma\big(\beta[(\log\pi_w - \log\pi^{\text{ref}}_w) - (\log\pi_l - \log\pi^{\text{ref}}_l)]\big)$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B8.1 à B8.6)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres ensemble, entre guillemets, dans n'importe quel ordre (`"EF"`) ; une forme : entre guillemets (`"(3, 5)"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B8.7, l'estimation 🧮 B8.8, la réflexion (🗣️ B8.9, ⚖️ B8.10, 📄 B8.11) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math
from itertools import product

import numpy as np


def sigmoid(u):
    return 1.0 / (1.0 + math.exp(-u))


def softplus(u):                                       # -log sigmoid(-u) = log(1 + e^u)
    return math.log1p(math.exp(u))


# quiz Q1: CartPole discretised into 6 intervals per variable, 2 actions ; a Q network on a batch of 32 states
Q1A = 6 ** 4 * 2
# quiz Q3: a full memory of 10 000 transitions, one replaced per step, 64 distinct transitions drawn per update
CAP_Q3, BATCH_Q3 = 10_000, 64
Q3A = CAP_Q3 * BATCH_Q3 / CAP_Q3                       # steps of stay x probability of being drawn at each step
Q3D = (1 - 0.005) ** 200                                # weight left on the target weights of 200 steps ago
# quiz Q4: three actions of value 0, estimates +1 or -1 with probability 1/2, independently
OUTCOMES_Q4 = list(product([-1, 1], repeat=3))
Q4B = float(np.mean([max(o) for o in OUTCOMES_Q4]))
Q4C = float(np.mean([second[int(np.argmax(first))] for first in OUTCOMES_Q4 for second in OUTCOMES_Q4]))
# quiz Q5: 6 motors with 10 values each ; a softmax with logits (ln 2, ln 6, 0)
Q5B = 10 ** 6
LOGITS_Q5 = np.array([math.log(2), math.log(6), 0.0])
Q5C = float(np.exp(LOGITS_Q5)[1] / np.exp(LOGITS_Q5).sum())
# quiz Q6: a biased coin, p(heads) = theta = 0.3 ; the score function d/dtheta log p(x)
THETA_Q6 = 0.3
SCORE_TAILS_Q6 = -1 / (1 - THETA_Q6)
Q6B = THETA_Q6 * (1 / THETA_Q6) + (1 - THETA_Q6) * SCORE_TAILS_Q6
# quiz Q7: one TD error ; the weight of delta_(t+10) with gamma = 0.99 and lambda = 0.95
Q7A = 1 + 0.9 * 2.5 - 3.0
Q7D = (0.99 * 0.95) ** 10
# quiz Q8: N = 8 environments, T = 256 steps, 4 epochs, mini-batches of 256 ; a ratio from two log-probabilities
Q8C = 4 * (8 * 256) // 256
Q8D = math.exp(-0.9 - (-1.2))
# quiz Q10: an evaluation cut at max_steps = 300 ; the standard error of a mean of 100 returns
Q10A, Q10B = 300, 40 / math.sqrt(100)
# quiz Q11: GRPO advantages in a group (1, 0, 0, 0), population standard deviation
REWARDS_Q11 = np.array([1.0, 0.0, 0.0, 0.0])
Q11C = float((REWARDS_Q11[0] - REWARDS_Q11.mean()) / REWARDS_Q11.std())
Q11C_DDOF1 = float((REWARDS_Q11[0] - REWARDS_Q11.mean()) / REWARDS_Q11.std(ddof=1))
# recall R1: one Q-learning update
Q_R1, R_R1, GAMMA_R1, MAX_R1, ALPHA_R1 = 0.5, 0.3, 0.8, 0.6, 0.2
R1A = R_R1 + GAMMA_R1 * MAX_R1
R1B = Q_R1 + ALPHA_R1 * (R1A - Q_R1)
R1C = Q_R1 + ALPHA_R1 * (R_R1 - Q_R1)
# recall R2: the first step of Adam with bias correction ; clipping by the global norm (PyTorch adds 1e-6)
W_R2, G_R2, LR_R2, B1_R2, B2_R2, EPS_R2 = 1.0, -40.0, 1e-3, 0.9, 0.999, 1e-8
M_R2, V_R2 = (1 - B1_R2) * G_R2, (1 - B2_R2) * G_R2 ** 2
R2A = W_R2 - LR_R2 * (M_R2 / (1 - B1_R2)) / (math.sqrt(V_R2 / (1 - B2_R2)) + EPS_R2)
R2A_NO_CORRECTION = W_R2 - LR_R2 * M_R2 / (math.sqrt(V_R2) + EPS_R2)
GRADS_R2 = [np.array([3.0, 4.0]), np.array([12.0])]
NORM_R2 = math.sqrt(sum(float((g ** 2).sum()) for g in GRADS_R2))
R2B = 3.0 * min(1.0, 0.5 / (NORM_R2 + 1e-6))
# recall R3: epsilon-greedy over 2 actions ; a softmax with temperature 0.5 ; an incremental mean
R3A = 1 - 0.2 + 0.2 / 2
R3B = float(np.exp(1.5 / 0.5) / (np.exp(1.0 / 0.5) + np.exp(1.5 / 0.5)))
R3C = 4 + (8 - 4) / 4

# ✏️ B8.1: three transitions, gamma = 0.99, r = 1 ; transition 2 is terminated, transition 3 truncated
GAMMA_B81 = 0.99
QT_B81 = np.array([[12.0, 13.7], [4.0, 3.1], [40.2, 41.6]])        # Q_target(s', .)
QO_B81 = np.array([[12.6, 12.2], [2.5, 3.8], [42.5, 41.7]])        # Q_online(s', .)
TERM_B81 = np.array([False, True, False])
PRED_B81 = np.array([15.0, 2.9, 45.3])                              # Q_online(s, a) of the played action
DQN_B81 = 1 + GAMMA_B81 * (1 - TERM_B81) * QT_B81.max(axis=1)
DDQN_B81 = 1 + GAMMA_B81 * (1 - TERM_B81) * QT_B81[np.arange(3), QO_B81.argmax(axis=1)]
GAP_B81 = float(np.mean(DQN_B81 - DDQN_B81))
AGREE_B81 = bool(QT_B81[0].argmax() == QO_B81[0].argmax())
X3_B81 = float(PRED_B81[2] - DQN_B81[2])

# ✏️ B8.2: three gaps x = Q - y, Huber threshold 1.5
X_B82, DELTA_B82 = np.array([0.6, -2.4, 3.2]), 1.5


def huber_b82(x, delta=DELTA_B82):
    x = np.asarray(x, float)
    return np.where(np.abs(x) <= delta, 0.5 * x ** 2, delta * (np.abs(x) - delta / 2))


def huber_grad_b82(x, delta=DELTA_B82):
    return np.clip(np.asarray(x, float), -delta, delta)


TERMS_B82 = huber_b82(X_B82)
MEAN_B82 = float(TERMS_B82.mean())
GRAD3_B82 = float(huber_grad_b82(X_B82)[2] / 3)
GRAD3_MSE_B82 = float(X_B82[2] / 3)
SMOOTH_B82 = MEAN_B82 / DELTA_B82                                   # smooth_l1_loss(beta) = huber(delta=beta) / beta
SAME_GRAD_B82 = bool(huber_grad_b82(0.6) == 0.6)

# ✏️ B8.3: softmax policy with logits (0.5, 1.5, -0.5), action 2 played, advantage -2, step 0.1
Z_B83, A_B83, ADV_B83, STEP_B83 = np.array([0.5, 1.5, -0.5]), 2, -2.0, 0.1


def softmax_b83(z):
    e = np.exp(np.asarray(z, float) - np.max(z))
    return e / e.sum()


PI_B83 = softmax_b83(Z_B83)
GRAD_B83 = np.eye(3)[A_B83] - PI_B83                                # grad_z log softmax(z)_a = e_a - softmax(z)
Z_NEW_B83 = Z_B83 + STEP_B83 * ADV_B83 * GRAD_B83
PI_NEW_B83 = softmax_b83(Z_NEW_B83)
ENTROPY_B83 = float(-(PI_B83 * np.log(PI_B83)).sum())

# ✏️ B8.4: four steps, gamma = 0.9, truncated at step 1 (no terminated step), the rollout ends after step 3
R_B84 = np.ones(4)
V_B84 = np.array([4.0, 3.8, 2.0, 2.4])
NEXT_B84 = np.array([3.8, 3.6, 2.4, 2.6])
TERM_B84 = np.zeros(4, dtype=bool)
TRUNC_B84 = np.array([False, True, False, False])
GAMMA_B84 = 0.9


def gae_b84(lam, terminated=TERM_B84, truncated=TRUNC_B84, values=V_B84, next_values=NEXT_B84):
    """Backward loop of 208.4: delta_t = r + gamma (1 - term) V(s') - V(s), A_t = delta_t + gamma lam (1 - done) A_(t+1)."""
    deltas = R_B84 + GAMMA_B84 * (1 - terminated) * next_values - values
    adv, future = np.zeros(4), 0.0
    for t in range(3, -1, -1):
        future = deltas[t] + GAMMA_B84 * lam * (1 - (terminated[t] or truncated[t])) * future
        adv[t] = future
    return deltas, adv


DELTA_B84, ADV0_B84 = gae_b84(0.0)
_, ADV95_B84 = gae_b84(0.95)
_, ADV1_B84 = gae_b84(1.0)
RET95_B84 = ADV95_B84 + V_B84
_, ADV95_AS_END_B84 = gae_b84(0.95, terminated=TRUNC_B84, truncated=TERM_B84)    # truncation taken as a true end
_, ADV95_NO_STOP_B84 = gae_b84(0.95, truncated=TERM_B84)                          # the recursion does not stop
MC_B84 = bool(math.isclose(ADV1_B84[0] + V_B84[0], 1 + GAMMA_B84 * 1 + GAMMA_B84 ** 2 * 3.6))

# ✏️ B8.5: four transitions, eps = 0.2 (then 0.3)
RATIO_B85 = np.array([1.4, 0.6, 0.75, 1.35])
ADV_B85 = np.array([2.0, 1.5, -1.2, -0.8])


def clip_b85(eps):
    unclipped = RATIO_B85 * ADV_B85
    clipped = np.clip(RATIO_B85, 1 - eps, 1 + eps) * ADV_B85
    terms = np.minimum(unclipped, clipped)
    grad = np.where(unclipped <= clipped, -ADV_B85 / 4, 0.0)
    return terms, -float(terms.mean()), grad


TERMS_B85, LOSS_B85, GRAD_B85 = clip_b85(0.2)
ZERO_B85 = int((GRAD_B85 == 0).sum())
CLIPFRAC_B85 = float(np.mean(np.abs(RATIO_B85 - 1) > 0.2))
ZERO_EPS3_B85 = int((clip_b85(0.3)[2] == 0).sum())
# a classic mistake for d): the max instead of the min
LOSS_MAX_B85 = -float(np.maximum(RATIO_B85 * ADV_B85, np.clip(RATIO_B85, 0.8, 1.2) * ADV_B85).mean())

# ✏️ B8.6: Bradley-Terry on one pair, then DPO
RW_B86, RL_B86 = 2.3, 1.1
P_B86 = sigmoid(RW_B86 - RL_B86)
LOSS_B86 = softplus(-(RW_B86 - RL_B86))
GRAD_W_B86 = -(1 - P_B86)
LOSS_SWAP_B86 = softplus(RW_B86 - RL_B86)
LOGP_B86 = {"pc": -35.2, "pr": -38.9, "rc": -36.0, "rr": -37.5}


def dpo_b86(beta):
    chosen = LOGP_B86["pc"] - LOGP_B86["rc"]
    rejected = LOGP_B86["pr"] - LOGP_B86["rr"]
    margin = beta * (chosen - rejected)
    return beta * chosen, margin, softplus(-margin), -beta * sigmoid(-margin)


IMPLICIT_B86, MARGIN_B86, DPO_B86, DPO_GRAD_B86 = dpo_b86(0.1)
DPO5_B86 = dpo_b86(0.5)[2]

# The same answers from the libraries (no download, no training): the reference mylearn.deep_rl and mylearn.rl,
# PyTorch (losses, autograd, Adam, clip_grad_norm_, Categorical) and gymnasium (CartPole-v1, seeds)
import warnings

import gymnasium as gym
import torch
import torch.nn.functional as F

DRL, RL = mylearn.deep_rl, mylearn.rl

# quiz Q1: CartPole-v1 has 4 state variables and 2 actions ; a Q network rends one value per action
env = gym.make("CartPole-v1")
assert env.observation_space.shape == (4,) and env.action_space.n == 2
assert env.spec.max_episode_steps == 500 and env.spec.reward_threshold == 475.0
assert Q1A == 6 ** env.observation_space.shape[0] * env.action_space.n
torch.manual_seed(0)
q_net = torch.nn.Sequential(torch.nn.Linear(4, 16), torch.nn.ReLU(), torch.nn.Linear(16, 2))
assert tuple(q_net(torch.zeros(32, 4)).shape) == (32, 2)

# quiz Q1 d): the semi-gradient (target detached) and the full gradient differ
s, s_next = torch.randn(1, 4), torch.randn(1, 4)
semi = (q_net(s)[0, 0] - (1 + 0.9 * q_net(s_next).max().detach())) ** 2
full = (q_net(s)[0, 0] - (1 + 0.9 * q_net(s_next).max())) ** 2
g_semi = torch.autograd.grad(semi, list(q_net.parameters()))
g_full = torch.autograd.grad(full, list(q_net.parameters()))
assert any((a - b).abs().max() > 1e-6 for a, b in zip(g_semi, g_full))

# quiz Q3 a): simulate the full memory of rl.ReplayBuffer, 64 distinct transitions per update
buffer = RL.ReplayBuffer(capacity=1_000, obs_shape=(4,), obs_dtype="float32", random_state=0)
for i in range(1_000):
    buffer.push(np.zeros(4, np.float32), 0, float(i), np.zeros(4, np.float32), False)
draws = np.zeros(1_000)
for _ in range(2_000):
    batch = buffer.sample(64)
    assert len(np.unique(batch["rewards"])) == 64                     # distinct transitions within a mini-batch
    draws[batch["rewards"].astype(int)] += 1
assert abs(draws.mean() * 1_000 / 2_000 - BATCH_Q3) < 1               # per 1 000 steps of stay: 64 draws
buffer.push(np.ones(4, np.float32), 1, -1.0, np.ones(4, np.float32), True)
assert len(buffer) == 1_000 and -1.0 in buffer.sample(1_000)["rewards"] and 0.0 not in buffer.sample(1_000)["rewards"]

# quiz Q4: Monte Carlo check of the exact enumerations
rng = np.random.default_rng(0)
est = rng.choice([-1, 1], size=(200_000, 2, 3))
assert abs(est[:, 0].max(axis=1).mean() - Q4B) < 0.01
assert abs(est[np.arange(200_000), 1, est[:, 0].argmax(axis=1)].mean() - Q4C) < 0.01

# quiz Q5 c), quiz R3 b): softmax by torch
assert math.isclose(torch.softmax(torch.tensor(LOGITS_Q5), 0)[1].item(), Q5C, rel_tol=1e-9)
assert math.isclose(torch.softmax(torch.tensor([1.0, 1.5], dtype=torch.float64) / 0.5, 0)[1].item(), R3B)

# quiz Q6: the score of a Bernoulli by autograd
theta = torch.tensor(THETA_Q6, dtype=torch.float64, requires_grad=True)
torch.log(1 - theta).backward()
assert math.isclose(theta.grad.item(), SCORE_TAILS_Q6) and abs(Q6B) < 1e-12

# quiz Q7: one TD error and the GAE weight with the reference compute_gae (lambda = 0 gives the TD error)
adv, _ = DRL.compute_gae([1.0], [3.0], [2.5], [False], [False], gamma=0.9, lam=0.0)
assert math.isclose(adv[0], Q7A)
impulse = np.zeros(11)
impulse[10] = 1.0                                  # a TD error of 1 at step 10 only: r = delta, V = 0
adv, _ = DRL.compute_gae(impulse, np.zeros(11), np.zeros(11), np.zeros(11, bool), np.zeros(11, bool), gamma=0.99, lam=0.95)
assert math.isclose(adv[0], Q7D)

# quiz Q8 f), ✏️ B8.5: the clipped objective by torch.min/torch.clamp and autograd, and the reference
for eps, (terms, loss, grad) in ((0.2, clip_b85(0.2)), (0.3, clip_b85(0.3))):
    ratio = torch.tensor(RATIO_B85, requires_grad=True)
    adv_t = torch.tensor(ADV_B85)
    loss_t = -torch.min(ratio * adv_t, ratio.clamp(1 - eps, 1 + eps) * adv_t).mean()
    loss_t.backward()
    assert math.isclose(loss_t.item(), loss) and np.allclose(ratio.grad.numpy(), grad)
    ref_loss, ref_grad = DRL.clipped_surrogate(RATIO_B85, ADV_B85, clip_eps=eps)
    assert math.isclose(ref_loss, loss) and np.allclose(ref_grad, grad)
_, grad_zero_adv = DRL.clipped_surrogate([0.3, 1.0, 2.5], [0.0, 0.0, 0.0])
assert np.all(grad_zero_adv == 0)

# quiz Q10: gymnasium seeds ; evaluate_policy cut at max_steps (a perfect rule of 208.1)
env.reset(seed=3)
first = env.reset(seed=3)[0]
assert not np.allclose(env.reset()[0], first)                     # reset() without a seed continues the sequence
env.reset(seed=3)
a1 = [env.action_space.sample() for _ in range(20)]
env.reset(seed=3)
a2 = [env.action_space.sample() for _ in range(20)]
assert a1 != a2                                                   # reset(seed=...) does not seed the action space
rule = lambda state, info: int(state[2] + 0.5 * state[3] > 0)    # noqa: E731
assert RL.evaluate_policy(env, rule, n_episodes=5, max_steps=Q10A, rng=np.random.default_rng(1))["mean_return"] == Q10A

# quiz Q11: the GRPO advantages of a group
assert math.isclose(Q11C, 3 / math.sqrt(3)) and math.isclose(Q11C_DDOF1, 1.5)

# recall R2: Adam and clip_grad_norm_ in PyTorch
w = torch.nn.Parameter(torch.tensor([W_R2], dtype=torch.float64))
opt = torch.optim.Adam([w], lr=LR_R2, betas=(B1_R2, B2_R2), eps=EPS_R2)
w.grad = torch.tensor([G_R2], dtype=torch.float64)
opt.step()
assert math.isclose(w.item(), R2A, rel_tol=1e-12)
p1, p2 = torch.nn.Parameter(torch.zeros(2)), torch.nn.Parameter(torch.zeros(1))
p1.grad, p2.grad = torch.tensor([3.0, 4.0]), torch.tensor([12.0])
assert math.isclose(torch.nn.utils.clip_grad_norm_([p1, p2], max_norm=0.5).item(), NORM_R2)
assert math.isclose(p1.grad[0].item(), R2B, rel_tol=1e-6)
assert math.isclose(float(torch.cat([p1.grad, p2.grad]).norm()), 0.5, rel_tol=1e-5)

# ✏️ B8.1: the targets in PyTorch (max, then argmax + gather), under no_grad
with torch.no_grad():
    qt, qo = torch.tensor(QT_B81), torch.tensor(QO_B81)
    term = torch.tensor(TERM_B81)
    y = 1 + GAMMA_B81 * (1 - term.double()) * qt.max(dim=1).values
    y_double = 1 + GAMMA_B81 * (1 - term.double()) * qt.gather(1, qo.argmax(dim=1, keepdim=True)).squeeze(1)
assert np.allclose(y.numpy(), DQN_B81) and np.allclose(y_double.numpy(), DDQN_B81)

# ✏️ B8.2: F.huber_loss and F.smooth_l1_loss, values and gradients
pred = torch.tensor(X_B82, requires_grad=True)
target = torch.zeros(3, dtype=torch.float64)
huber = F.huber_loss(pred, target, delta=DELTA_B82)
huber.backward()
assert math.isclose(huber.item(), MEAN_B82) and math.isclose(pred.grad[2].item(), GRAD3_B82)
assert np.allclose(F.huber_loss(pred.detach(), target, delta=DELTA_B82, reduction="none").numpy(), TERMS_B82)
assert math.isclose(F.smooth_l1_loss(pred.detach(), target, beta=DELTA_B82).item(), SMOOTH_B82)
mse = pred.detach().clone().requires_grad_()
(0.5 * mse ** 2).mean().backward()
assert math.isclose(mse.grad[2].item(), GRAD3_MSE_B82)

# ✏️ B8.3: log_softmax and autograd, one step, Categorical's entropy
z = torch.tensor(Z_B83, requires_grad=True)
torch.log_softmax(z, dim=0)[A_B83].backward()
assert np.allclose(z.grad.numpy(), GRAD_B83)
assert np.allclose(torch.softmax(torch.tensor(Z_NEW_B83), 0).numpy(), PI_NEW_B83)
assert math.isclose(torch.distributions.Categorical(logits=torch.tensor(Z_B83)).entropy().item(), ENTROPY_B83)

# ✏️ B8.4: the reference compute_gae for the three lambdas, and the two classic mistakes
for lam, expected in ((0.0, ADV0_B84), (0.95, ADV95_B84), (1.0, ADV1_B84)):
    adv, ret = DRL.compute_gae(R_B84, V_B84, NEXT_B84, TERM_B84, TRUNC_B84, gamma=GAMMA_B84, lam=lam)
    assert np.allclose(adv, expected) and np.allclose(ret, expected + V_B84)
adv, _ = DRL.compute_gae(R_B84, V_B84, NEXT_B84, TRUNC_B84, TERM_B84, gamma=GAMMA_B84, lam=0.95)
assert np.allclose(adv, ADV95_AS_END_B84)

# ✏️ B8.6: the reference losses and F.logsigmoid with autograd
loss, g_w, g_l = DRL.bradley_terry_loss([RW_B86], [RL_B86])
assert math.isclose(loss, LOSS_B86) and math.isclose(g_w[0], GRAD_W_B86) and math.isclose(g_l[0], -GRAD_W_B86)
assert math.isclose(DRL.bradley_terry_loss([RW_B86 + 10], [RL_B86 + 10])[0], LOSS_B86, rel_tol=1e-9)
assert math.isclose(DRL.bradley_terry_loss([RL_B86], [RW_B86])[0], LOSS_SWAP_B86)
pc = torch.tensor([LOGP_B86["pc"]], dtype=torch.float64, requires_grad=True)
for beta, expected in ((0.1, DPO_B86), (0.5, DPO5_B86)):
    loss_d, margins = DRL.dpo_loss([LOGP_B86["pc"]], [LOGP_B86["pr"]], [LOGP_B86["rc"]], [LOGP_B86["rr"]], beta=beta)
    assert math.isclose(loss_d, expected)
margin_t = 0.1 * ((pc - LOGP_B86["rc"]) - (LOGP_B86["pr"] - LOGP_B86["rr"]))
(-F.logsigmoid(margin_t)).mean().backward()
assert math.isclose(pc.grad.item(), DPO_GRAD_B86) and math.isclose(margin_t.item(), MARGIN_B86)
env.close()
print("✅ the paper answers agree with mylearn.deep_rl, mylearn.rl, PyTorch and gymnasium")

✅ the paper answers agree with mylearn.deep_rl, mylearn.rl, PyTorch and gymnasium


**B8.Q1 — Pourquoi une table Q ne suffit plus pour CartPole**

In [3]:
wb.record("B8.Q1a", Q1A, mistakes={"une case par état discrétisé et par action : n'oublie pas les actions": 6 ** 4,
          "les intervalles des quatre variables se combinent : ils se multiplient, ils ne s'additionnent pas": 48,
          "6 intervalles pour chacune des 4 variables : vérifie la base et l'exposant": 4 ** 6 * 2,
          "relis la question : 6 intervalles par variable, et quatre variables": 24})
wb.record("B8.Q1b", 1, mistakes={"relis ce que rend le réseau de 208.1 : une valeur pour un couple (état, action), ou un vecteur ?": 2,
          "il faut bien calculer les valeurs pour choisir : relis « Pourquoi la table ne suffit plus » (§208.1)": 0})
wb.record("B8.Q1c", True, mistakes={"relis « Pourquoi la table ne suffit plus » (§208.1) : que partagent deux états voisins dans un réseau ?": False})
wb.record("B8.Q1d", False, mistakes={"relis « Apprendre une valeur par régression » (§208.1) : que devient la méthode si la cible est dérivée aussi ?": True})
wb.record("B8.Q1e", "(32, 2)", mistakes={"le réseau rend une valeur par action : combien d'actions a CartPole ?": "(32, 1)",
          "4 est la dimension de l'entrée : que rend le réseau ?": "(32, 4)",
          "relis la question : un batch de 32 états passe d'un coup, pas un seul état": "(2,)",
          "la forme commence par l'axe des exemples": "(2, 32)"})

WB_ANSWER {"hash": "d2fa5a37ad88529b6d0be6d384471d428548b2c7bd4f9a9a8a66151d0670fd3d", "id": "B8.Q1a", "kind": "int", "magnitude": 3, "mistakes": {"30fd59084e4b088d912ea30af167c8546d7b614a99ebc07561bd41a01ffdfc8f": "une case par état discrétisé et par action : n'oublie pas les actions", "9d4f016d4beaeb1ab2a37ea4fe3c68e3186f48c11457aeb58ae8f924f7c33317": "les intervalles des quatre variables se combinent : ils se multiplient, ils ne s'additionnent pas", "d310ca425a3d252a219fe6736169238c20e88306f825fff358b0c745c8e773b4": "relis la question : 6 intervalles par variable, et quatre variables", "f72571d3ea2c6ee99dbc554700dbed511840c1ffcc39777f6330a291c0a5fb4c": "6 intervalles pour chacune des 4 variables : vérifie la base et l'exposant"}, "sign": 1}
📝 Ex B8.Q1a : réponse enregistrée (int).
WB_ANSWER {"hash": "13c28f05d5fc79a998845407e2f5cbd7b7c0de1e6547569629c06d215a474536", "id": "B8.Q1b", "kind": "int", "magnitude": 0, "mistakes": {"88ca780ed2dd1acc60bddf06801bf0252b62e46a0d06a25f470be74c6

**B8.Q2 — La triade mortelle : approximation, amorçage et apprentissage off-policy**

In [4]:
wb.record("B8.Q2a", "ACD", mistakes={"il manque au moins une lettre : relis « Instabilité : la triade mortelle » (§208.1)": {'C', 'CD', 'AD', 'D', 'AC', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis « Instabilité : la triade mortelle » (§208.1)": {'ABCDE', 'ABCD', 'ACDE'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis « Instabilité : la triade mortelle » (§208.1)": {'BD', 'ABE', 'ABD', 'ABDE', 'DE', 'ACE', 'BCDE', 'ADE', 'BCD', 'BDE', 'BC', 'BE', 'ABCE', 'BCE', 'ABC', 'AB', 'E', 'B', 'CDE', 'CE', 'AE'}})
wb.record("B8.Q2b", 2, mistakes={"une table est-elle une approximation de fonction ? Relis la triade (§208.1)": 3,
          "relis la cible du Q-learning (ch. 26) : amorce-t-elle ? Et apprend-elle la valeur de la politique qui explore ?": 1,
          "compte chaque ingrédient séparément : approximation, amorçage, off-policy": 0})
wb.record("B8.Q2c", 1, mistakes={"des retours observés jusqu'à la fin d'un épisode : y a-t-il une estimation dans la cible ?": 2,
          "un réseau qui estime V est-il une approximation de fonction ?": 0,
          "relis les trois ingrédients : les transitions viennent-elles d'une autre politique ?": 3})
wb.record("B8.Q2d", 2, mistakes={"SARSA apprend la valeur de la politique qu'il suit (ch. 26) : est-ce off-policy ?": 3,
          "la cible de SARSA contient Q(s', a') : est-ce un amorçage ? Et un réseau, une approximation ?": 1,
          "compte chaque ingrédient séparément : approximation, amorçage, off-policy": 0})
wb.record("B8.Q2e", False, mistakes={"relis la fin de « Instabilité : la triade mortelle » (§208.1) : DQN renonce-t-il à l'un des trois ?": True})
wb.record("B8.Q2f", True, mistakes={"relis le troisième ingrédient (§208.1) : de quelle politique viennent les transitions de la mémoire ?": False})

WB_ANSWER {"hash": "0d0aa6cd1868543159c78e36e26b846e6238a4c0ef6b5d82e8d86b600d14b4cc", "id": "B8.Q2a", "kind": "str", "mistakes": {"140f221fee457f71db71d17fb043a92dc0ea384e861f1b7bc2417888cc56db31": "il manque au moins une lettre, et une autre n'a pas sa place : relis « Instabilité : la triade mortelle » (§208.1)", "18f06a5eb269408bea690252c17b1fcb7165911e1a300f36e917021b99633e96": "il manque au moins une lettre : relis « Instabilité : la triade mortelle » (§208.1)", "22cac5f1cd24d54268048d8a1f468b4a40b9cf16dabada34e14e398adafc278b": "au moins une des lettres choisies n'a pas sa place : relis « Instabilité : la triade mortelle » (§208.1)", "24ed11aa4d17fd3ad455b7eb1a5472e4bac5653bc981e6479104ed34ddfb8961": "au moins une des lettres choisies n'a pas sa place : relis « Instabilité : la triade mortelle » (§208.1)", "28c66432e0cdef45f696d929f1a6eb927235693ada951db7c10830582f2c4dbb": "il manque au moins une lettre, et une autre n'a pas sa place : relis « Instabilité : la triade mortelle » (

**B8.Q3 — Réseau cible et mémoire de rejeu : quel problème chacun résout-il ?**

In [5]:
wb.record("B8.Q3a", round(Q3A), mistakes={"une mise à jour tire 64 transitions, pas une seule : ta probabilité d'être tirée à chaque pas en tient-elle compte ?": 1,
          "relis la question : combien de fois une même transition est-elle tirée, pas combien de mini-batches tient la mémoire": 156,
          "une transition reste dans la mémoire plus d'un pas : combien de pas, et quelle chance d'être tirée à chacun ?": 0},
          fractional="compte les tirages d'une transition pendant tout son séjour : un nombre entier en moyenne ici")
wb.record("B8.Q3b", False, mistakes={"refais le calcul de a) avec 1 000 : la durée du séjour et la probabilité d'être tirée changent toutes les deux": True})
wb.record("B8.Q3c", True, mistakes={"relis « Les deux stabilisateurs » (§208.2) : quels poids calculent la cible entre deux copies ?": False})
wb.record("B8.Q3d", Q3D, decimals=2, mistakes={"les parts se multiplient d'un pas à l'autre, elles ne se retranchent pas": 0.0,
          "relis la question : 200 pas, pas un seul": 0.995,
          "relis la question : la part des poids d'il y a 200 pas qui reste, pas celle de ce qui a été ajouté depuis": 1 - Q3D})
wb.record("B8.Q3e", True, mistakes={"relis « Les deux stabilisateurs » (§208.2) : combien de fois sert chaque transition de la mémoire ?": False})

WB_ANSWER {"fractional": "compte les tirages d'une transition pendant tout son séjour : un nombre entier en moyenne ici", "hash": "0ab77d41892dfa45ebe2aebcf6eef896b528e5be6e9e0e4f8235341c7c724d50", "id": "B8.Q3a", "kind": "int", "magnitude": 1, "mistakes": {"3f06b79c65509de91d7203204cea7c87c2f1d8429cd99e01d043851bed5c6a42": "une mise à jour tire 64 transitions, pas une seule : ta probabilité d'être tirée à chaque pas en tient-elle compte ?", "970e3e1cdd6ce7a18765e891100b7f855c019fe9be1f2b653b3f35620e8f6356": "relis la question : combien de fois une même transition est-elle tirée, pas combien de mini-batches tient la mémoire", "d4ee8d1a04948066577cb296ab8310adafb461e7b05417dbb7ba538b4fb4fcb5": "une transition reste dans la mémoire plus d'un pas : combien de pas, et quelle chance d'être tirée à chacun ?"}, "sign": 1}
📝 Ex B8.Q3a : réponse enregistrée (int).
WB_ANSWER {"hash": "90cc7240c6adba6b59a4c5a9358facfbedfdd22c26a4a7fc4a7179e01154dbf1", "id": "B8.Q3b", "kind": "bool", "mistakes": {

**B8.Q4 — Double DQN : d'où vient la surestimation des valeurs Q ?**

In [6]:
wb.record("B8.Q4a", "B", mistakes={"chaque estimation est juste en moyenne, mais leur maximum ? Relis « La surestimation, et Double DQN » (§208.2)": "A", "le max choisit l'estimation la plus haute : dans quel sens penche-t-il ?": "C"})
wb.record("B8.Q4b", Q4B, decimals=2, mistakes={"chaque estimation est juste en moyenne, mais le maximum des trois ?": 0.0,
          "compte les huit cas de trois estimations, pas les quatre cas de deux": 0.5,
          "le maximum vaut −1 dans un cas : lequel, et avec quelle probabilité ?": 1.0,
          "relis la question : l'espérance du maximum, pas la probabilité qu'il vaille +1": 0.875})
wb.record("B8.Q4c", round(Q4C), mistakes={"le bruit qui a fait gagner l'action dans le premier jeu est-il celui qu'on lit dans le second ?": 1,
          "les deux jeux sont indépendants : relis « La surestimation, et Double DQN » (§208.2)": -1},
          fractional="la valeur lue ne profite plus du bruit qui a fait gagner l'action : son espérance est un entier ici")
wb.record("B8.Q4d", "A", mistakes={"relis la formule de Double DQN (§208.2) : quel réseau prend l'argmax ?": "B", "Double DQN n'ajoute aucun réseau : relis la fin de « La surestimation, et Double DQN »": "C"})
wb.record("B8.Q4e", False, mistakes={"relis la fin de « La surestimation, et Double DQN » (§208.2) : combien de paramètres le changement ajoute-t-il ?": True})
wb.record("B8.Q4f", True, mistakes={"la politique gloutonne prend l'argmax : que lui fait une même constante ajoutée à toutes les valeurs ?": False})

WB_ANSWER {"hash": "37847f32439eb9d6dec2de6f5742f47e43b2c89a823edc62297372128dc538bf", "id": "B8.Q4a", "kind": "str", "mistakes": {"540e066f30f4ec6aaf5d00569a177b51771c552488cfb3b1a82f4bedb036c130": "le max choisit l'estimation la plus haute : dans quel sens penche-t-il ?", "830af59c0f6fbba9f6a6ac0451ecde9ed733123b41ac413cfdaf7ea55e677f77": "chaque estimation est juste en moyenne, mais leur maximum ? Relis « La surestimation, et Double DQN » (§208.2)"}}
📝 Ex B8.Q4a : réponse enregistrée (str).
WB_ANSWER {"decimals": 2, "hash": "a59dc9b18a46fb76c948ae716233240f878e04510ffc10623710f089d72b1eef", "hash_coarse": "8d7f55e1beacba85ce6c1b1b9f0e76dc783688b97c7fe294be53f3b986493574", "id": "B8.Q4b", "kind": "float", "magnitude": -1, "mistakes": {"65a44f5b0683174e602f13d0e51b8ef70bcd7ef6ad72f893cbc93a333683cfda": "relis la question : l'espérance du maximum, pas la probabilité qu'il vaille +1", "de7722eea6f2c515b1320d8f8a926679cb1c21b74bbc3fced57f49160ac1169c": "compte les huit cas de trois estim

**B8.Q5 — Méthodes par valeur ou par politique : actions continues et politiques aléatoires**

In [7]:
wb.record("B8.Q5a", "C", mistakes={"calcule b) : combien de colonnes aurait cette table, et que perd-on entre deux valeurs d'un moteur ?": "A", "un DQN prend un argmax sur une liste d'actions : relis la première raison de « Apprendre la politique elle-même » (§208.3)": "B"})
wb.record("B8.Q5b", Q5B, mistakes={"les choix des six moteurs se combinent : ils se multiplient": 60,
          "relis : 10 valeurs pour chacun des 6 moteurs (vérifie la base et l'exposant)": 6 ** 10,
          "relis la question : 10 valeurs possibles par moteur": 2 ** 6})
wb.record("B8.Q5c", Q5C, decimals=2, mistakes={"c'est la probabilité d'une autre action : relis la numérotation à partir de 0": 2 / 9,
          "la softmax passe par l'exponentielle des logits, pas par les logits eux-mêmes": math.log(6) / (math.log(2) + math.log(6)),
          "relis la question : trois actions, et l'exponentielle de 0 vaut 1": 0.75,
          "trois logits différents ne donnent pas trois probabilités égales": 1 / 3})
wb.record("B8.Q5d", "ABD", mistakes={"il manque au moins une lettre : relis « Apprendre la politique elle-même » (§208.3) et la mémoire de rejeu (§208.2)": {'AB', 'BD', 'B', 'D', 'AD', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis « Apprendre la politique elle-même » (§208.3) et la mémoire de rejeu (§208.2)": {'ABCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis « Apprendre la politique elle-même » (§208.3) et la mémoire de rejeu (§208.2)": {'C', 'ABC', 'ACD', 'BC', 'CD', 'AC', 'BCD'}})
wb.record("B8.Q5e", False, mistakes={"si ta politique est déterministe, que fait un adversaire qui l'a devinée ? Relis la deuxième raison de « Apprendre la politique elle-même » (§208.3)": True})

WB_ANSWER {"hash": "3c4cc7e12cbcfec7abb6dc40df4e27d36397db651c5ccd7b213400b115ad34f1", "id": "B8.Q5a", "kind": "str", "mistakes": {"77002c747651795c17489eda989c9beb00c6f6f4cbb5cadbc9c36467887bb74c": "un DQN prend un argmax sur une liste d'actions : relis la première raison de « Apprendre la politique elle-même » (§208.3)", "7a0f3c58b0689b2b14f6b044141e5dc5d766d054f73b99f7c191582c64c0844e": "calcule b) : combien de colonnes aurait cette table, et que perd-on entre deux valeurs d'un moteur ?"}}
📝 Ex B8.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "ef534a8d9b751970c1acded11fc6f74f77d7751cf878d5221c261d36ae7d9662", "id": "B8.Q5b", "kind": "int", "magnitude": 6, "mistakes": {"0b5b6639794901c3d4fd3e08dbc95a5992487f535df3363240420247a04ac57a": "relis la question : 10 valeurs possibles par moteur", "0cfbed4c23f1a3a95e4ef368303924c1ed4c76c957446d865b407122bb9c5229": "les choix des six moteurs se combinent : ils se multiplient", "a64cbed8e28db4bcb7d9b963b9455af5556e346f09a19f6ab7db3d2e109

**B8.Q6 — REINFORCE : pourquoi soustraire une baseline ne biaise pas le gradient**

In [8]:
wb.record("B8.Q6a", SCORE_TAILS_Q6, decimals=2, mistakes={"c'est le score de pile : relis la question": 1 / THETA_Q6,
          "vérifie le signe : ln(1 − θ) baisse quand θ monte": -SCORE_TAILS_Q6,
          "dérive ln(1 − θ), pas ln θ": -1 / THETA_Q6})
wb.record("B8.Q6b", Q6B, decimals=2, mistakes={"pondère chaque issue par sa probabilité": 1 / THETA_Q6 + SCORE_TAILS_Q6,
          "relis la question : l'espérance sur les deux issues, pas le score d'une seule": SCORE_TAILS_Q6})
wb.record("B8.Q6c", "C", mistakes={"une moyenne d'épisodes passés dépend-elle de l'action jouée maintenant ? Relis « La baseline et la variance » (§208.3)": "A", "un critique dépend de l'état, pas de l'action : relis « La baseline et la variance » (§208.3)": "B"})
wb.record("B8.Q6d", True, mistakes={"une constante ne dépend pas de l'action : que devient l'espérance ? Et (G − b)², quand b est loin des retours ?": False})
wb.record("B8.Q6e", False, mistakes={"relis la fin de « REINFORCE » (§208.3) : l'estimation de REINFORCE a-t-elle un biais à corriger ?": True})

WB_ANSWER {"decimals": 2, "hash": "eca824caab7cfe7a2ed225449dfab4b80ce0eda846a364dbb7d2292c499c0ae7", "hash_coarse": "c35035e61a2c6f512d8b17a67c14af2d22cd6682dcd981a7af666dc294828fa6", "id": "B8.Q6a", "kind": "float", "magnitude": 0, "mistakes": {"9324794d40697fb771d1119d7690249c5eec47a06a354ea104082571d2e896cc": "c'est le score de pile : relis la question", "d6331dc6caf9596dc85d590e91434fc4ef917bdcf336d6bbe9cf3899ce1c8552": "dérive ln(1 − θ), pas ln θ", "fd3da39c97fa7edb5496495bd03895f243cf80a7a5eeb443e11da0a87fdfc9ca": "vérifie le signe : ln(1 − θ) baisse quand θ monte"}, "sign": -1}
📝 Ex B8.Q6a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "f7bd168d1fbebc6d0db9e3e5cb88af21654d077b22b62bf33fe2335e0cf0bc2f", "hash_coarse": "61c8022cd72da67df00cae57bb8e9611c8a573a2e9f4b948012371c4f36f4285", "id": "B8.Q6b", "kind": "float", "magnitude": null, "mistakes": {"a342fb9f40c9e4a679aae5925b06110d65232b3eea03c94c56f4937939bf24c1": "relis la question : l'espéranc

**B8.Q7 — Avantage, erreur TD et GAE : ce que règle λ**

In [9]:
wb.record("B8.Q7a", Q7A, decimals=2, mistakes={"γ multiplie la valeur de l'état suivant": 1 + 2.5 - 3.0,
          "l'épisode continue : la cible amorce sur V(s_{t+1})": 1 - 3.0,
          "l'erreur TD retranche la valeur de l'état actuel": 1 + 0.9 * 2.5,
          "vérifie le signe : la cible moins la valeur prévue": -Q7A})
wb.record("B8.Q7b", "A", mistakes={"relis les cas particuliers de GAE (§208.4) : quel λ redonne G_t − V(s_t) ?": "B", "relis la formule de GAE (§208.4) : que reste-t-il de la somme quand γλ = 0 ?": "C"})
wb.record("B8.Q7c", "C", mistakes={"relis les cas particuliers de GAE (§208.4) : quel réglage porte le hasard de tout le reste de l'épisode ?": "A", "λ = 0 ne garde qu'un pas de hasard : relis « L'erreur TD, première estimation de l'avantage »": "B"})
wb.record("B8.Q7d", Q7D, decimals=2, mistakes={"le poids combine γ et λ, à chaque pas": 0.95 ** 10,
          "le poids combine γ et λ, à chaque pas : λ manque": 0.99 ** 10,
          "relis la question : dix pas plus loin, pas un seul": 0.99 * 0.95,
          "vérifie l'exposant : δ_{t+10} est dix pas après δ_t": (0.99 * 0.95) ** 11})
wb.record("B8.Q7e", False, mistakes={"relis le cas λ = 1 (§208.4) : que reste-t-il du critique dans G_t − V(s_t) ?": True})
wb.record("B8.Q7f", True, mistakes={"relis « Fin d'épisode ou limite de temps » (§208.4) : la récursion continue-t-elle après un pas tronqué ?": False})

WB_ANSWER {"alt_coarse": ["428601344d41ede56fc6ea84751f6d022826cc360cbdbcbc4e610ad617f605fb"], "decimals": 2, "hash": "d666846928b7f02f895f1a912038a178af742127cece4667fa8b3913df0f3f9a", "hash_coarse": "1693c13ec06df28033337d664b310b0a854bd38959de7133e06359cc5125cee3", "id": "B8.Q7a", "kind": "float", "magnitude": -1, "mistakes": {"1165a39bf3807647a8b72ea43b3166b9adbe4d413f3c16895a925b315be75e06": "l'épisode continue : la cible amorce sur V(s_{t+1})", "a06b4294e1e0663336ac35ad799da82abb4ff054f80795d4ea2f9c6cc42cf584": "vérifie le signe : la cible moins la valeur prévue", "aa419b6627c540fbbaff96bbbda9c476a56cdd7983c7bd04008857889557610e": "l'erreur TD retranche la valeur de l'état actuel", "f25ec8aeba47a42f0fcbe3b0b20776ced5746d1faf2fbc6aa0f1f678d22390ee": "γ multiplie la valeur de l'état suivant"}, "sign": 1}
📝 Ex B8.Q7a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "3b5018ff2cfa6231110897ba3983c1057c778b882acb96c3f56ec63aa0f64952", "id": "B8.Q7b", "kind": "str", "mis

**B8.Q8 — PPO : à quoi sert le clip, et pourquoi réutiliser plusieurs fois le même lot ?**

In [10]:
wb.record("B8.Q8a", "B", mistakes={"relis « Réutiliser un rollout sans tout casser » (§208.5) : le clip rend-il l'estimation exacte ?": "A", "relis le début de « Réutiliser un rollout sans tout casser » (§208.5) : un gradient de politique est-il off-policy ?": "C"})
wb.record("B8.Q8b", False, mistakes={"relis le cas de l'avantage négatif (§208.5) : quel terme est le minimum quand le ratio est monté ?": True})
wb.record("B8.Q8c", Q8C, mistakes={"relis la question : 4 epochs, pas une seule": Q8C // 4,
          "chaque epoch découpe tout le rollout en mini-batches : combien en tient-il ?": 4,
          "relis la question : on demande des pas d'Adam, pas des transitions": 8 * 256})
wb.record("B8.Q8d", Q8D, decimals=2, mistakes={"vérifie le sens du ratio : la nouvelle politique sur l'ancienne": 1 / Q8D,
          "le ratio est l'exponentielle de la différence des log-probabilités": 0.3,
          "on divise des probabilités, pas des log-probabilités": 0.75})
wb.record("B8.Q8e", False, mistakes={"relis « Une région de confiance » (§208.5) : comment TRPO résout-il son problème ?": True})
wb.record("B8.Q8f", True, mistakes={"relis le gradient par rapport au ratio (§208.5) : que vaut −A/n quand A = 0 ?": False})

WB_ANSWER {"hash": "c0c7fffeeb39be3b52a599f7f1a843013348c264db2f5df5095d813007415bcd", "id": "B8.Q8a", "kind": "str", "mistakes": {"246c1c6b75a0a68a095ac040926748d1a017af98c0a8bfe9ae4ceb471b459a7a": "relis « Réutiliser un rollout sans tout casser » (§208.5) : le clip rend-il l'estimation exacte ?", "c99755ef4d0e3a16f3345bbc3cb14f7f2bd527ab1d74f54de3d4899ab6de6fb3": "relis le début de « Réutiliser un rollout sans tout casser » (§208.5) : un gradient de politique est-il off-policy ?"}}
📝 Ex B8.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "6c9e05ce913f77dc589d79551daf8b42ae6dcbb7f9384090a32cd0e192d5a991", "id": "B8.Q8b", "kind": "bool", "mistakes": {"15756c8b84145a550bf54bbb75ee353579f46c1bf09f9f15a429a03ca2732767": "relis le cas de l'avantage négatif (§208.5) : quel terme est le minimum quand le ratio est monté ?"}}
📝 Ex B8.Q8b : réponse enregistrée (bool).
WB_ANSWER {"hash": "5d14b064c091a5e8f96d3833204303cf4eca19876dc7a683d7ec1e62e10eb728", "id": "B8.Q8c", "kind": "int", "magnit

**B8.Q9 — On-policy ou off-policy : DQN, REINFORCE et PPO**

In [11]:
wb.record("B8.Q9a", "B", mistakes={"relis la fin de « REINFORCE » (§208.3) : un épisode sert-il encore après le pas ?": "A", "PPO corrige l'écart par un ratio : relis « Réutiliser un rollout sans tout casser » (§208.5)": "C"})
wb.record("B8.Q9b", False, mistakes={"relis l'encadré 💼 de 208.5 : quelle famille consomme beaucoup d'interactions, et pourquoi ?": True})
wb.record("B8.Q9c", True, mistakes={"relis la fin de « REINFORCE » (§208.3) : pour quelle politique le gradient vaut-il ?": False})
wb.record("B8.Q9d", "A", mistakes={"le ratio de probabilités appartient à PPO : relis la cible de DQN (§208.2)": "B", "même avec ε = 0,1, la cible reste celle de la politique gloutonne : relis la cible de DQN (§208.2)": "C"})
wb.record("B8.Q9e", "C", mistakes={"relis l'encadré 💼 de 208.5 : que coûte PPO en interactions, ici où chacune prend du temps réel ?": "A", "une estimation sans biais mais très dispersée demande beaucoup d'épisodes : relis « La baseline et la variance » (§208.3)": "B"})

WB_ANSWER {"hash": "121ab4387789a8492e7b71836e8a2c5cda67a23fb59dccb9075c47db4d599ba6", "id": "B8.Q9a", "kind": "str", "mistakes": {"2da6d51117d3ac88db23386a976e6bd5927aa784ecbe7475b06112ba9e9ac425": "relis la fin de « REINFORCE » (§208.3) : un épisode sert-il encore après le pas ?", "b76cf330b6199d9d2ef2f8485158181390d2775bfb78ad9c9403569dd71c66ac": "PPO corrige l'écart par un ratio : relis « Réutiliser un rollout sans tout casser » (§208.5)"}}
📝 Ex B8.Q9a : réponse enregistrée (str).
WB_ANSWER {"hash": "3bb327d869ab76c608eda78d4ce77298575ff83738d869255325b37b9ea0d55b", "id": "B8.Q9b", "kind": "bool", "mistakes": {"d3dc298d8c719f0e923ae86ed19e96bb8dc48435ae890770f47d6c898b518d7c": "relis l'encadré 💼 de 208.5 : quelle famille consomme beaucoup d'interactions, et pourquoi ?"}}
📝 Ex B8.Q9b : réponse enregistrée (bool).
WB_ANSWER {"hash": "f9b17c26694b67f290ff7080635de1d936b7f1a4f2ddfe1921b90c51a7244f20", "id": "B8.Q9c", "kind": "bool", "mistakes": {"f8442743e1cb24a4dc8dca00a8b4f9af04e1648

**B8.Q10 — Évaluer un agent : graines, épisodes, politique déterministe**

In [12]:
wb.record("B8.Q10a", Q10A, mistakes={"relis l'encadré ⚠️ « un plafond caché dans l'évaluation » (§208.6) : que fait max_steps ?": 500,
          "relis l'appel : max_steps n'a pas sa valeur par défaut ici": 200,
          "relis la question : une politique parfaite, qui ne tombe jamais": 475})
wb.record("B8.Q10b", Q10B, decimals=1, mistakes={"relis la question : l'erreur type de la moyenne, pas l'écart-type des retours": 40.0,
          "l'erreur type ne divise pas l'écart-type par le nombre d'épisodes : relis le ch. 2": 0.4})
wb.record("B8.Q10c", False, mistakes={"relis « Les graines » (§208.6) : que font les reset() suivants, sans graine ?": True})
wb.record("B8.Q10d", "B", mistakes={"plus d'épisodes d'évaluation réduisent le bruit de l'évaluation, pas celui de l'entraînement : relis « Les graines » (§208.6)": "A", "garder la meilleure graine, c'est choisir un tirage favorable : relis « Rapporter plusieurs graines » (§208.6)": "C"})
wb.record("B8.Q10e", True, mistakes={"relis « Les graines » (§208.6) : l'espace d'actions a-t-il son propre générateur ?": False})
wb.record("B8.Q10f", "A", mistakes={"relis « Évaluer un agent » (§208.6) : quelle politique évalue-t-on, et sans quel hasard ?": "B", "V dépend de l'état seul : peut-il choisir une action ?": "C"})

WB_ANSWER {"hash": "b36430aeb04c607fafd98c70d6875f0a38d9c5d25ce56191ecc1fedd02c3efbb", "id": "B8.Q10a", "kind": "int", "magnitude": 2, "mistakes": {"09237e2aece28fddfbed32487f397459412dc7c179c0ffefd31a30e05513c43a": "relis l'appel : max_steps n'a pas sa valeur par défaut ici", "b8fb9d1cea478a7296a070c8dcbd1806a2d5ee192e144aeadb8af7cb63aab1f1": "relis l'encadré ⚠️ « un plafond caché dans l'évaluation » (§208.6) : que fait max_steps ?", "cb48d90df213eb8ceacc72559c882a6ee06b7d9fd7ba783170b283edd1f2a375": "relis la question : une politique parfaite, qui ne tombe jamais"}, "sign": 1}
📝 Ex B8.Q10a : réponse enregistrée (int).
WB_ANSWER {"decimals": 1, "hash": "b41a1243209fa378f74e2ced5bc6f2e8795d9f65430469701bcb80dfceec70bb", "hash_coarse": "622889d20ef39413406ff9edcd74c053cb8cdd70bcfd4ff84ab8907f0a8acad6", "id": "B8.Q10b", "kind": "float", "magnitude": 0, "mistakes": {"17a22be6631c69e8f8a8a554a22cf52638d7ac2de954e25aabf25f3c45fd5407": "relis la question : l'erreur type de la moyenne, pas l'

**B8.Q11 — RLHF, DPO, GRPO : qui a besoin d'un modèle de récompense ou d'un critique ?**

In [13]:
wb.record("B8.Q11a", "A", mistakes={"au moins une des lettres choisies n'a pas sa place : relis 208.7 : DPO apprend-il une récompense à part, et d'où vient celle de GRPO ?": {'AC', 'ABC', 'AB'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis 208.7 : DPO apprend-il une récompense à part, et d'où vient celle de GRPO ?": {'C', 'BC', 'B'}})
wb.record("B8.Q11b", "AC", mistakes={"il manque au moins une lettre : relis 208.7 : lesquelles sont des méthodes de RL, et laquelle est une loss supervisée ?": {'C', 'A'}, "au moins une des lettres choisies n'a pas sa place : relis 208.7 : lesquelles sont des méthodes de RL, et laquelle est une loss supervisée ?": {'ABC'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis 208.7 : lesquelles sont des méthodes de RL, et laquelle est une loss supervisée ?": {'AB', 'BC', 'B'}})
wb.record("B8.Q11c", Q11C, decimals=2, mistakes={"relis la question : l'écart-type de population, divisé par le nombre de réponses": Q11C_DDOF1,
          "l'avantage divise l'écart à la moyenne par l'écart-type du groupe": 0.75,
          "relis la question : l'avantage de la première réponse, celle qui a la récompense 1": -1 / math.sqrt(3)})
wb.record("B8.Q11d", True, mistakes={"relis le dernier piège du tableau de la fiche : π_ref bouge-t-elle ?": False})
wb.record("B8.Q11e", "B", mistakes={"une récompense 0 ou 1 varie d'une réponse à l'autre : relis « GRPO et les récompenses vérifiables » (§208.7)": "A", "relis « GRPO et les récompenses vérifiables » (§208.7) : que coûte le critique de PPO ?": "C"})
wb.record("B8.Q11f", True, mistakes={"calcule les avantages d'un groupe de récompenses toutes égales : que pousse le gradient ?": False})

WB_ANSWER {"hash": "56fa5eb299b0dbba107d38d52061df7d3ee9843396dd2c468938982df5678f4d", "id": "B8.Q11a", "kind": "str", "mistakes": {"02a5d98de0da64e6a780813d7678d47fc1873f62f8ca702075f595e2a5c054df": "au moins une des lettres choisies n'a pas sa place : relis 208.7 : DPO apprend-il une récompense à part, et d'où vient celle de GRPO ?", "2403b10ca6373d88fb5c66a9f3230fc219b79cdcc04a0a046ac49de896166886": "il manque au moins une lettre, et une autre n'a pas sa place : relis 208.7 : DPO apprend-il une récompense à part, et d'où vient celle de GRPO ?", "870f04294670b351467284f90c51ed3c2c3f063a31e3c428c88149d9b4f9c7bd": "au moins une des lettres choisies n'a pas sa place : relis 208.7 : DPO apprend-il une récompense à part, et d'où vient celle de GRPO ?", "8b69f75625da79ff844b65f60000b36da4eecb5703908e3262439996f57bbb5b": "au moins une des lettres choisies n'a pas sa place : relis 208.7 : DPO apprend-il une récompense à part, et d'où vient celle de GRPO ?", "9c78c42b306169516555148e691f122dd

**B8.Q12 — Piratage de la récompense (reward hacking) : quand l'agent optimise la lettre et pas l'esprit**

In [14]:
wb.record("B8.Q12a", "BD", mistakes={"il manque au moins une lettre : relis « Le piratage de la récompense » (§208.7) : l'agent atteint-il la récompense sans faire ce qu'on voulait ?": {'D', 'B'}, "au moins une des lettres choisies n'a pas sa place : relis « Le piratage de la récompense » (§208.7) : l'agent atteint-il la récompense sans faire ce qu'on voulait ?": {'ABCD', 'ABD', 'BCD'}, "il manque au moins une lettre, et une autre n'a pas sa place : relis « Le piratage de la récompense » (§208.7) : l'agent atteint-il la récompense sans faire ce qu'on voulait ?": {'C', 'ABC', 'AB', 'ACD', 'BC', 'CD', 'AD', 'AC', 'A'}})
wb.record("B8.Q12b", "B", mistakes={"relis « Le piratage de la récompense » (§208.7) : que fait la vraie récompense après un moment ?": "A", "relis « Le piratage de la récompense » (§208.7) : la vraie récompense monte-t-elle d'abord ?": "C"})
wb.record("B8.Q12c", False, mistakes={"relis « Le RLHF : PPO sous une laisse KL » (§208.7) : que fait une laisse plus forte ?": True})
wb.record("B8.Q12d", "C", mistakes={"plus d'optimisation contre le même modèle aggrave-t-elle ou corrige-t-elle le piratage ? Relis la courbe de Gao et coll.": "A", "relis le rôle de la pénalité KL (§208.7) : que se passe-t-il loin des réponses de la référence ?": "B"})
wb.record("B8.Q12e", True, mistakes={"relis le paragraphe sur la flagornerie (§208.7) : d'où viennent les préférences pour une réponse flatteuse ?": False})

WB_ANSWER {"hash": "e470854ae25d5f2e8910b7cfd4ccc5d03881bb3e00108b7c8f97ee3219ab46b0", "id": "B8.Q12a", "kind": "str", "mistakes": {"19c4d20a626d3adc1885e2305fc1e081eb9abdcc632344340bde95f86e975095": "au moins une des lettres choisies n'a pas sa place : relis « Le piratage de la récompense » (§208.7) : l'agent atteint-il la récompense sans faire ce qu'on voulait ?", "202dc24a0be20d5fade616fd1b26c7ebbd176bc529134fecea95e347f1288b8d": "il manque au moins une lettre, et une autre n'a pas sa place : relis « Le piratage de la récompense » (§208.7) : l'agent atteint-il la récompense sans faire ce qu'on voulait ?", "2e5fdd534effb66dce04cca02d788f352904093f08c0069746fdcca7d262b7fb": "il manque au moins une lettre, et une autre n'a pas sa place : relis « Le piratage de la récompense » (§208.7) : l'agent atteint-il la récompense sans faire ce qu'on voulait ?", "3516b7efe7556433dab924785edb150d7bef0e3adcfc038b42479ff057970e7c": "il manque au moins une lettre : relis « Le piratage de la récompense

**B8.R1 — La mise à jour du Q-learning et le rôle de la mémoire de rejeu**

In [15]:
wb.record("B8.R1a", R1A, decimals=2, mistakes={"la cible prend le maximum de la ligne de s', pas la valeur de la case actuelle": R_R1 + GAMMA_R1 * Q_R1,
          "s' n'est pas terminal : la cible amorce sur la ligne de s'": R_R1,
          "γ multiplie la valeur de la suite": R_R1 + MAX_R1})
wb.record("B8.R1b", R1B, decimals=3, mistakes={"α mélange la cible et l'ancienne valeur : relis la mise à jour du ch. 26 (§26.6.1)": R1A,
          "relis la question : la nouvelle valeur de la case, pas l'erreur TD": R1A - Q_R1,
          "relis la question : la nouvelle valeur de la case, pas seulement sa variation": ALPHA_R1 * (R1A - Q_R1)})
wb.record("B8.R1c", R1C, decimals=2, mistakes={"un état terminal n'a pas de suite : relis le rappel maths du ch. 26 (§26.6.1)": R1B,
          "α mélange la cible et l'ancienne valeur": R_R1})
wb.record("B8.R1d", "B", mistakes={"c'est le rejeu du livre : relis l'encadré 🕰️ de 26.9 et la docstring de ReplayBuffer": "A", "relis la docstring de ReplayBuffer (🔨 26.31) : que range push ?": "C"})
wb.record("B8.R1e", True, mistakes={"relis la docstring de ReplayBuffer (🔨 26.31) : que devient la transition la plus ancienne ?": False})
wb.record("B8.R1f", False, mistakes={"relis le ⚠️ du ch. 26 sur SARSA (§26.7) : de quelle politique SARSA apprend-il la valeur ?": True})

WB_ANSWER {"decimals": 2, "hash": "15ec256e28bb1ddb8e96783ac64438eeaa2cf4801f4e917c906fb7f74dd1c882", "hash_coarse": "6e31ddac038df001826aac948fad6f8f6445b6606c2ec1a806b5e7f40f4dfb8a", "id": "B8.R1a", "kind": "float", "magnitude": -1, "mistakes": {"9cdb8c36470502c4e7e76770bb5a01510d5607a17fc3fdd46e2fee141972b69b": "γ multiplie la valeur de la suite", "c131dba18954bcd3167fcf31caa3559250f261aea6bf9ac395e51e20c5679a63": "s' n'est pas terminal : la cible amorce sur la ligne de s'", "e3e003b3efa22f244d16086cb962c1d6a00265b6f0fdfe7321236c5967449104": "la cible prend le maximum de la ligne de s', pas la valeur de la case actuelle"}, "sign": 1}
📝 Ex B8.R1a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "01c63f68b25181a2485f4924898aac3cf4ac9e8c1e6b19948a9bbfe53233e2e0", "hash_coarse": "1efe128d941c61f94f0ad49e2277604f1c85a7636bb59b60c464d3eed89b2fd1", "id": "B8.R1b", "kind": "float", "magnitude": -1, "mistakes": {"2abb0b33a3fbeb834da0fe02e7ac65ad9fc445fefaa8ef0a

**B8.R2 — Adam et le gradient clipping**

In [16]:
wb.record("B8.R2a", R2A, decimals=4, mistakes={"c'est le pas de la descente de gradient simple : relis Adam au ch. 19 (§19.5.5)": W_R2 - LR_R2 * G_R2,
          "vérifie le signe : le gradient est négatif, le poids monte": 2 * W_R2 - R2A,
          "relis la correction de biais d'Adam (ch. 19, §19.5.5) : m et v partent de 0": R2A_NO_CORRECTION})
wb.record("B8.R2b", R2B, decimals=3, mistakes={"la norme est globale : tous les gradients mis bout à bout, et un seul facteur": 3.0 * 0.5 / 5.0,
          "le clipping par la norme ne borne pas chaque composante à ±0,5": 0.5,
          "la norme globale dépasse 0,5 : le gradient est réduit": 3.0})
wb.record("B8.R2c", False, mistakes={"relis « le gradient clipping » (ch. 19) : à quel moment les gradients existent-ils, et quand sont-ils utilisés ?": True})
wb.record("B8.R2d", "A", mistakes={"relis la formule d'Adam (ch. 19, §19.5.5) : où ε apparaît-il ?": "B", "borner la norme est le rôle de max_grad_norm : relis la formule d'Adam": "C"})
wb.record("B8.R2e", True, mistakes={"relis « le gradient clipping » (ch. 19) : tous les gradients sont-ils multipliés par le même facteur ?": False})

WB_ANSWER {"decimals": 4, "hash": "05631aaffda7a75d7103adee423c37ff74fda203b05bc3ad430976a3ae52e324", "hash_coarse": "7af90db83d37790929b6a9337d9e20ab27497dd4e5281dca1297fc85aa7c3620", "id": "B8.R2a", "kind": "float", "magnitude": 0, "mistakes": {"146f7901695e303c4d4e2fdd239c2060609093249799638433e8eaaa77497464": "c'est le pas de la descente de gradient simple : relis Adam au ch. 19 (§19.5.5)", "446a614ab22d387ab0c490113edf90227e21af494babf69428f5c5de23b73660": "vérifie le signe : le gradient est négatif, le poids monte", "d89d636214d432e048ed2c4e99397e26ac7c85531d7ecfc0db6d0e6939bb77ad": "relis la correction de biais d'Adam (ch. 19, §19.5.5) : m et v partent de 0"}, "sign": 1}
📝 Ex B8.R2a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "2b4791b0904fea104e365ba812800dabf8f243f6d72fe24523a6827bb770e3c4", "hash_coarse": "446555837e96f517fdd8c67457002a34f4cee8f5fdae78c9e16c324e07c2f97a", "id": "B8.R2b", "kind": "float", "magnitude": -1, "mistakes": {"56b80f

**B8.R3 — Exploration, softmax et bandits**

In [17]:
wb.record("B8.R3a", R3A, decimals=2, mistakes={"en exploration, ε-greedy tire parmi toutes les actions, l'action gloutonne comprise": 0.8,
          "relis la question : la probabilité de jouer l'action gloutonne, pas d'explorer": 0.2,
          "relis la question : deux actions pour CartPole": 1 - 0.2 + 0.2 / 4})
wb.record("B8.R3b", R3B, decimals=3, mistakes={"la température divise les valeurs, elle ne les multiplie pas": 1 / (1 + math.exp(-0.25)),
          "relis la question : la température vaut 0,5, pas 1": 1 / (1 + math.exp(-0.5)),
          "la softmax passe par l'exponentielle, pas par Q / ΣQ": 0.6})
wb.record("B8.R3c", R3C, decimals=1, mistakes={"compte les récompenses reçues par ce bras, la nouvelle comprise : quel n ?": 4 + 4 / 3,
          "la moyenne incrémentale ajoute (R − Q)/n, elle ne remplace pas Q par R": 8.0,
          "relis la question : la nouvelle moyenne, après la récompense 8": 4.0})
wb.record("B8.R3d", False, mistakes={"relis la softmax du ch. 26 (§26.6.2) : grande température, presque uniforme ou presque gloutonne ?": True})
wb.record("B8.R3e", "B", mistakes={"ε-greedy ne monte aucune préférence : relis l'encadré 🔗 de 208.3": "A", "UCB ajoute un bonus d'incertitude : relis l'encadré 🔗 de 208.3": "C"})

WB_ANSWER {"decimals": 2, "hash": "108bc0431bae5782a7c91ba866b2305f1fd08362c8ec932e6569d910b6aa929a", "hash_coarse": "7562f524261e00f6753dfff6de965a87130155fb1bf87c6393db3282c7f8df94", "id": "B8.R3a", "kind": "float", "magnitude": -1, "mistakes": {"275e491820c961889c8164bd2515c0eae09a80e41bd4b23ba17e4f1fed8caab5": "en exploration, ε-greedy tire parmi toutes les actions, l'action gloutonne comprise", "33a89dce7f5db9055e265ade44852ca9c3686e1a7bcdd5b449b0d86a1f438991": "relis la question : deux actions pour CartPole", "70e3aa0151212dc016d27563122a10c5948cf733a39d0f75001b0fb77152bf18": "relis la question : la probabilité de jouer l'action gloutonne, pas d'explorer"}, "sign": 1}
📝 Ex B8.R3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "836acf4c5deb0d31789c5860eb637e23318beaf93685d874f1faf31019cbd1c8", "hash_coarse": "407d07060a568cb9fa4a7b142969f1a71e4d0d9ee3dc89d8262527c4d580f01e", "id": "B8.R3b", "kind": "float", "magnitude": -1, "mistakes": {"7f43af3351

**Ex B8.1 — Cible DQN à la main : état terminal, puis cible Double DQN**

In [18]:
wb.record("B8.1a", DQN_B81[0], decimals=2, mistakes={"DQN prend le max du réseau cible, pas du réseau en ligne": 1 + GAMMA_B81 * QO_B81[0].max(),
          "γ multiplie la valeur de la suite : n'oublie pas la récompense r = 1": GAMMA_B81 * QT_B81[0].max(),
          "relis la question : on demande la cible DQN ; quelle action de s' retient-elle ?": 1 + GAMMA_B81 * QT_B81[0, 0]})
wb.record("B8.1b", DQN_B81[1], decimals=2, mistakes={"relis « la cible se réduit à r » (§208.2) : s' est-il terminal ?": 1 + GAMMA_B81 * QT_B81[1].max(),
          "même terminale, la transition a reçu sa récompense": 0.0})
wb.record("B8.1c", DQN_B81[2], decimals=2, mistakes={"relis l'encadré ⚠️ « la limite de temps n'est pas une fin » (§208.2)": 1.0,
          "DQN prend le max du réseau cible, pas du réseau en ligne": 1 + GAMMA_B81 * QO_B81[2].max(),
          "γ multiplie la valeur de la suite : n'oublie pas la récompense r = 1": GAMMA_B81 * QT_B81[2].max()})
wb.record("B8.1d", DDQN_B81[0], decimals=2, mistakes={"Double DQN lit la valeur dans le réseau cible, pas dans le réseau en ligne": 1 + GAMMA_B81 * QO_B81[0, 0],
          "relis la formule de Double DQN (§208.2) : quel réseau choisit l'action ?": 1 + GAMMA_B81 * QT_B81[0, 1]})
wb.record("B8.1e", DDQN_B81[2], decimals=2, mistakes={"Double DQN lit la valeur dans le réseau cible, pas dans le réseau en ligne": 1 + GAMMA_B81 * QO_B81[2, 0],
          "relis la formule de Double DQN (§208.2) : quel réseau choisit l'action ?": 1 + GAMMA_B81 * QT_B81[2, 1],
          "relis l'encadré ⚠️ « la limite de temps n'est pas une fin » (§208.2)": 1.0})
wb.record("B8.1f", GAP_B81, decimals=2, mistakes={"relis la question : la moyenne sur les trois transitions, la transition 2 comprise": GAP_B81 * 3 / 2,
          "vérifie le sens : la cible de DQN moins celle de Double DQN": -GAP_B81,
          "relis la question : la moyenne, pas la somme": GAP_B81 * 3})
wb.record("B8.1g", AGREE_B81, mistakes={"prends l'argmax de chacun des deux réseaux, puis compare": not AGREE_B81})
wb.record("B8.1h", X3_B81, decimals=2, mistakes={"vérifie le sens : la prédiction moins la cible": -X3_B81,
          "relis la question : l'écart à la cible DQN, pas à celle de Double DQN": float(PRED_B81[2] - DDQN_B81[2]),
          "relis l'encadré ⚠️ « la limite de temps n'est pas une fin » (§208.2) : que vaut la cible de la transition 3 ?": float(PRED_B81[2] - 1.0)})

WB_ANSWER {"decimals": 2, "hash": "32572da77da782676ccb7ee70316252ef3e9586bced625a276a1fa1cec88fc2b", "hash_coarse": "02b9e5904a8106f2b0dc25b63011b4e5fea0866ac32c5fc9b1fc81107b8b68de", "id": "B8.1a", "kind": "float", "magnitude": 1, "mistakes": {"0f98787a3419ccd2033d6a6453b73693a70a0f06517b0bf013c1b9054492c954": "γ multiplie la valeur de la suite : n'oublie pas la récompense r = 1", "a3323d3ed86eaeaab607ec1db1f1377dcf86e2a021fc886b65db58c3753e2e5f": "DQN prend le max du réseau cible, pas du réseau en ligne", "a8502818a26ff7796c64f8ef47880bc6130f62cc62efd4770f80b2c9e68964b7": "relis la question : on demande la cible DQN ; quelle action de s' retient-elle ?"}, "sign": 1}
📝 Ex B8.1a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "5fd072be759f4289f7d45293b1f4ca68bca3d4dec904b262b81e9570e37f8dec", "hash_coarse": "1f12cbda99aa464cf0d94beddb133954b8ef110b21f68676b400f089ee56d380", "id": "B8.1b", "kind": "float", "magnitude": 0, "mistakes": {"20996a8807b06573bc

**Ex B8.2 — Loss de Huber et son gradient pour trois erreurs TD**

In [19]:
wb.record("B8.2a", float(TERMS_B82[0]), decimals=2, mistakes={"|0,6| est sous le seuil : relis la branche quadratique, avec son ½": 0.6 ** 2,
          "|0,6| est sous le seuil : la branche linéaire ne s'applique pas": DELTA_B82 * (0.6 - DELTA_B82 / 2)})
wb.record("B8.2b", float(TERMS_B82[1]), decimals=3, mistakes={"|−2,4| dépasse le seuil : relis la branche linéaire": 0.5 * 2.4 ** 2,
          "la branche linéaire retranche δ/2 à |x|, pas δ": DELTA_B82 * (2.4 - DELTA_B82),
          "vérifie le signe : la loss de Huber prend la valeur absolue": DELTA_B82 * (-2.4 - DELTA_B82 / 2),
          "la branche linéaire multiplie par δ": 2.4 - DELTA_B82 / 2})
wb.record("B8.2c", MEAN_B82, decimals=2, mistakes={"relis l'énoncé : la moyenne des trois termes, pas leur somme": float(TERMS_B82.sum()),
          "c'est la moyenne des ½x² : relis le seuil": float((0.5 * X_B82 ** 2).mean()),
          "tu as divisé par δ : relis l'énoncé, F.huber_loss": MEAN_B82 / DELTA_B82})
wb.record("B8.2d", float(huber_grad_b82(-2.4)), decimals=1, mistakes={"au-delà du seuil, la dérivée ne dépend plus de x : relis le rappel maths de 208.2": -2.4,
          "vérifie le signe : x est négatif": DELTA_B82,
          "c'est la dérivée de smooth_l1_loss : relis la question, ℓ_δ": -1.0})
wb.record("B8.2e", GRAD3_B82, decimals=2, mistakes={"la loss du mini-batch est une moyenne : divise par 3": float(huber_grad_b82(3.2)),
          "au-delà du seuil, la dérivée ne dépend plus de x": 3.2 / 3,
          "vérifie le signe : x = Q − y, et Q monte avec x": -GRAD3_B82})
wb.record("B8.2f", GRAD3_MSE_B82, decimals=3, mistakes={"la loss du mini-batch est une moyenne : divise par 3": 3.2,
          "la dérivée de ½x² vaut x, pas 2x": 2 * 3.2 / 3,
          "relis la question : la moyenne des ½x², sans seuil": GRAD3_B82})
wb.record("B8.2g", SMOOTH_B82, decimals=3, mistakes={"relis le rappel maths de 208.2 : smooth_l1_loss(beta) et huber_loss(delta) ne coïncident qu'en 1": MEAN_B82,
          "smooth_l1_loss divise la loss de Huber par β, elle ne la multiplie pas": MEAN_B82 * DELTA_B82,
          "relis l'énoncé : la moyenne, pas la somme": float(TERMS_B82.sum()) / DELTA_B82})
wb.record("B8.2h", SAME_GRAD_B82, mistakes={"|0,6| est-il sous le seuil 1,5 ? Compare les deux dérivées dans ce cas": not SAME_GRAD_B82})

WB_ANSWER {"decimals": 2, "hash": "c0ec5ac196cf368edc2726d0ca5a9a2241ff9f46ba82d52087f1fcbb7e5f3aea", "hash_coarse": "53fff6fd0bf2ab965951d166e95cb6a1b8a48748252a76c6b4bbb3b7b85047ca", "id": "B8.2a", "kind": "float", "magnitude": -1, "mistakes": {"1b281158b261bcebefd68b0af4b87fe4d976906d1553752c3e2cff33fd339ed3": "|0,6| est sous le seuil : la branche linéaire ne s'applique pas", "6aad6098b75ac180446f0da83181942046662ec9f04dec8fbd82f61aeb4aa310": "|0,6| est sous le seuil : relis la branche quadratique, avec son ½"}, "sign": 1}
📝 Ex B8.2a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"alt_coarse": ["7105ea1a43e276ad9b197523d66592805bc4bf02276f69e062a3c3643190ddd2"], "decimals": 3, "hash": "d3e93d0f44868a4fae30f77c250a74ada99997df4c8d429aa6502444bee8f372", "hash_coarse": "33fc58d1d07458109d717ba221a7c516e8e66992dc566626386a8743c79fa1b1", "id": "B8.2b", "kind": "float", "magnitude": 0, "mistakes": {"1892bd418ef530ec85d964ccc429110ea387f0cb81f76e6e256e2fd12344976f": "|−2,4| dépas

**Ex B8.3 — Gradient du log d'une politique softmax à trois actions**

In [20]:
wb.record("B8.3a", float(PI_B83[2]), decimals=4, mistakes={"la softmax normalise les exponentielles par leur somme": float(np.exp(-0.5)),
          "c'est la probabilité d'une autre action : relis la numérotation à partir de 0": float(PI_B83[0]),
          "les logits ne sont pas des probabilités : passe par l'exponentielle": -0.5 / 1.5})
wb.record("B8.3b", float(PI_B83[1]), decimals=4, mistakes={"c'est la probabilité d'une autre action : relis la numérotation à partir de 0": float(PI_B83[2]),
          "la softmax normalise les exponentielles par leur somme": float(np.exp(1.5))})
wb.record("B8.3c", float(GRAD_B83[0]), decimals=4, mistakes={"vérifie le signe : e_a − softmax(z), et z_0 n'est pas l'action jouée": float(PI_B83[0]),
          "relis la formule de 208.3 : e_a vaut 0 en position 0 ici, l'action jouée est la 2": float(1 - PI_B83[0]),
          "relis la question : la composante de z_0": float(GRAD_B83[2])})
wb.record("B8.3d", float(GRAD_B83[2]), decimals=4, mistakes={"relis la formule de 208.3 : e_a vaut 1 en position a": float(-PI_B83[2]),
          "relis la question : la dérivée de log π(2), pas celle de π(2)": float(PI_B83[2] * (1 - PI_B83[2]))})
wb.record("B8.3e", True, mistakes={"additionne e_a − softmax(z) composante par composante : que vaut la somme de la softmax ?": False})
wb.record("B8.3f", float(Z_NEW_B83[2]), decimals=3, mistakes={"vérifie le signe : l'avantage est négatif, le logit de l'action jouée doit baisser": float(Z_B83[2] - STEP_B83 * ADV_B83 * GRAD_B83[2]),
          "relis l'énoncé : le pas multiplie aussi l'avantage": float(Z_B83[2] + STEP_B83 * GRAD_B83[2]),
          "relis l'énoncé : le pas vaut 0,1": float(Z_B83[2] + ADV_B83 * GRAD_B83[2])})
wb.record("B8.3g", float(PI_NEW_B83[2]), decimals=4, mistakes={"relis la question : après le pas, avec les trois nouveaux logits": float(PI_B83[2]),
          "les trois logits bougent : refais la softmax avec les trois nouveaux": float(softmax_b83([0.5, 1.5, Z_NEW_B83[2]])[2]),
          "vérifie le signe du pas : l'avantage est négatif": float(softmax_b83(Z_B83 - STEP_B83 * ADV_B83 * GRAD_B83)[2])})
wb.record("B8.3h", ENTROPY_B83, decimals=3, mistakes={"l'entropie de la fiche est en nats : le logarithme naturel": float(-(PI_B83 * np.log2(PI_B83)).sum()),
          "vérifie le signe : H = −Σ π ln π est positive": -ENTROPY_B83,
          "relis la question : l'entropie avant le pas": float(-(PI_NEW_B83 * np.log(PI_NEW_B83)).sum())})

WB_ANSWER {"decimals": 4, "hash": "7d7cd8388ce92425173f47354411e4eddd02b9c62e725e9a91a73a0d51624ff5", "hash_coarse": "9c715087552e759c8cb272a1b876b968e3c4983ac2d3a0c70d9389e37239f41d", "id": "B8.3a", "kind": "float", "magnitude": -2, "mistakes": {"270eb07e933a84d59f3d88325879f7ec538f0945b5aacffc8caa537d9c7b0104": "les logits ne sont pas des probabilités : passe par l'exponentielle", "65894e2f80412a29da5404cafa9bb552f680bf0a70ccd641f9730107a6614a16": "la softmax normalise les exponentielles par leur somme", "65f328962dd95c44b0c7e99ff7323a1a54c2361c9f5326af6056aad82f0b2e77": "c'est la probabilité d'une autre action : relis la numérotation à partir de 0"}, "sign": 1}
📝 Ex B8.3a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "997efebaa3783a05ab41c7456605b511db6984071ad974ff88c699ab4dac8153", "hash_coarse": "629305c866cf45c8329e671948ec752540ca30d60b998ae08119fad5a321dd1f", "id": "B8.3b", "kind": "float", "magnitude": -1, "mistakes": {"20c5cd61943b1c62a0e58d

**Ex B8.4 — GAE à la main sur quatre pas, pour λ = 0, 0,95 et 1**

In [21]:
wb.record("B8.4a", float(DELTA_B84[1]), decimals=2, mistakes={"relis « Fin d'épisode ou limite de temps » (§208.4) : un pas tronqué amorce-t-il ?": float(1 - V_B84[1]),
          "next_values[1] est la valeur de la vraie dernière observation : relis l'énoncé": float(1 + GAMMA_B84 * V_B84[2] - V_B84[1]),
          "γ multiplie la valeur de l'état suivant": float(1 + NEXT_B84[1] - V_B84[1])})
wb.record("B8.4b", float(DELTA_B84[3]), decimals=2, mistakes={"le rollout s'arrête, pas l'épisode : relis la docstring de compute_gae (next_values)": float(1 - V_B84[3]),
          "γ multiplie la valeur de l'état suivant": float(1 + NEXT_B84[3] - V_B84[3])})
wb.record("B8.4c", float(ADV0_B84[2]), decimals=2, mistakes={"relis le cas λ = 0 (§208.4)": float(ADV1_B84[2]),
          "relis le cas λ = 0 (§208.4) : avantage et cible du critique sont deux sorties différentes": float(ADV0_B84[2] + V_B84[2])})
wb.record("B8.4d", float(ADV95_B84[2]), decimals=4, mistakes={"la récursion multiplie par γλ, pas par λ seul": float(DELTA_B84[2] + 0.95 * DELTA_B84[3]),
          "la récursion multiplie par γλ, pas par γ seul": float(DELTA_B84[2] + GAMMA_B84 * DELTA_B84[3]),
          "relis la question : λ = 0,95, l'erreur TD suivante compte aussi": float(DELTA_B84[2]),
          "relis la question : l'avantage, pas la cible du critique": float(RET95_B84[2])})
wb.record("B8.4e", float(ADV95_B84[0]), decimals=4, mistakes={"relis « Fin d'épisode ou limite de temps » (§208.4) : la récursion s'arrête-t-elle au pas 1 ?": float(ADV95_NO_STOP_B84[0]),
          "relis « Fin d'épisode ou limite de temps » (§208.4) : le pas tronqué amorce encore": float(ADV95_AS_END_B84[0]),
          "la récursion multiplie par γλ, pas par λ seul": float(DELTA_B84[0] + 0.95 * DELTA_B84[1])})
wb.record("B8.4f", float(RET95_B84[0]), decimals=4, mistakes={"relis la docstring de compute_gae : les cibles valent avantages + values": float(ADV95_B84[0] + NEXT_B84[0]),
          "relis la question : la cible du critique (avantage + valeur)": float(ADV95_B84[0])})
wb.record("B8.4g", float(ADV1_B84[0]), decimals=3, mistakes={"relis « Fin d'épisode ou limite de temps » (§208.4) : la récursion s'arrête-t-elle au pas 1 ?": float(gae_b84(1.0, truncated=TERM_B84)[1][0]),
          "relis la question : λ = 1": float(ADV95_B84[0]),
          "relis la question : l'avantage, pas la cible du critique": float(ADV1_B84[0] + V_B84[0])})
wb.record("B8.4h", float(ADV1_B84[2]), decimals=3, mistakes={"relis la question : λ = 1": float(ADV95_B84[2]),
          "le pas 3 n'est pas la fin de l'épisode : son erreur TD compte aussi": float(DELTA_B84[2])})
wb.record("B8.4i", float(ADV95_AS_END_B84[0]), decimals=3, mistakes={"relis la question : sans amorçage au pas 1, l'erreur TD de ce pas change": float(ADV95_B84[0]),
          "relis la question : λ = 0,95": float(gae_b84(1.0, terminated=TRUNC_B84, truncated=TERM_B84)[1][0])})
wb.record("B8.4j", MC_B84, mistakes={"écris Â_0 = δ_0 + γ δ_1 avec λ = 1, puis ajoute V(s_0) : les valeurs intermédiaires se simplifient-elles ?": not MC_B84})

WB_ANSWER {"decimals": 2, "hash": "d0daabeb219f05f5a3def9b3ac12fa6281dfa3f06ceaae13fc9f2c9dcdc35576", "hash_coarse": "7100509f33b121eaf4f47661c07f1e33641fad3d13538c155416a8a84e9fbe49", "id": "B8.4a", "kind": "float", "magnitude": -1, "mistakes": {"0ae4312f8aa2cfbe34bbb0cd0756846026a8f9a08c06e495c6a4fd16bc3b75ee": "relis « Fin d'épisode ou limite de temps » (§208.4) : un pas tronqué amorce-t-il ?", "4cf85a1886b0a985ff75fa93195e9eba377cd4fd535cd3374560ef8898eedc11": "γ multiplie la valeur de l'état suivant", "7accc9c201380db53f03c4ca1df9cbdd44a1faf0aec7603101adb66b0706d8af": "next_values[1] est la valeur de la vraie dernière observation : relis l'énoncé"}, "sign": 1}
📝 Ex B8.4a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "1805263bc6c6b30c6aa0b2b831cb973bac03eae3ed5d2d8a9f7d31a32e1b0eae", "hash_coarse": "bd9b629d0eb6aeb5ed71bae56b6c1a033a64e791c2fb2c161df8e3b8ebc1d6f1", "id": "B8.4b", "kind": "float", "magnitude": -1, "mistakes": {"0d35e01befee5243643a8

**Ex B8.5 — Objectif clippé de PPO : quatre cas, valeur et gradient nul ou non**

In [22]:
wb.record("B8.5a", float(TERMS_B85[0]), decimals=2, mistakes={"le min prend le plus petit des deux termes : relis le cas de l'avantage positif (§208.5)": float(RATIO_B85[0] * ADV_B85[0]),
          "le clip porte sur le ratio, pas sur le produit": 1.2})
wb.record("B8.5b", float(TERMS_B85[2]), decimals=2, mistakes={"le min prend le plus petit des deux termes : avec un avantage négatif, lequel est le plus petit ?": float(RATIO_B85[2] * ADV_B85[2]),
          "relis la borne basse de l'intervalle : 1 − ε": float(1.2 * ADV_B85[2])})
wb.record("B8.5c", float(TERMS_B85[3]), decimals=2, mistakes={"le min prend le plus petit des deux termes : avec un avantage négatif, lequel est le plus petit ?": float(1.2 * ADV_B85[3]),
          "vérifie le signe : l'avantage est négatif": float(-RATIO_B85[3] * ADV_B85[3])})
wb.record("B8.5d", LOSS_B85, decimals=3, mistakes={"vérifie le signe : la loss est l'opposé de l'objectif": -LOSS_B85,
          "relis l'énoncé : la moyenne des quatre termes, pas leur somme": LOSS_B85 * 4,
          "relis le rôle du min (§208.5) : il prend le plus pessimiste des deux termes": LOSS_MAX_B85})
wb.record("B8.5e", float(GRAD_B85[1]), decimals=3, mistakes={"vérifie le signe : la loss baisse si ce ratio monte": -float(GRAD_B85[1]),
          "le gradient de la loss moyenne divise par le nombre de transitions": -float(ADV_B85[1]),
          "compare les deux termes de la transition 1 : lequel est le minimum ?": 0.0})
wb.record("B8.5f", float(GRAD_B85[3]), decimals=2, mistakes={"compare les deux termes de la transition 3 : le non clippé est-il le minimum ?": 0.0,
          "vérifie le signe : la loss baisse si ce ratio redescend": -float(GRAD_B85[3]),
          "le gradient de la loss moyenne divise par le nombre de transitions": -float(ADV_B85[3])})
wb.record("B8.5g", ZERO_B85, mistakes={"un ratio hors de l'intervalle n'a pas toujours un gradient nul : compare les deux termes, transition par transition": 4,
          "compare les deux termes de chaque transition : où le terme clippé est-il strictement le plus petit ?": 1,
          "relis la figure 1 de l'article : dans quelles zones le gradient s'annule-t-il ?": 3})
wb.record("B8.5h", CLIPFRAC_B85, decimals=2, mistakes={"la fraction clippée compte les ratios hors de l'intervalle, pas les gradients nuls": ZERO_B85 / 4,
          "relis la définition : |ρ − 1| > ε, pour chaque transition": 0.75})
wb.record("B8.5i", ZERO_EPS3_B85, mistakes={"avec ε = 0,3, l'intervalle devient [0,7 ; 1,3] : refais la comparaison des deux termes": ZERO_B85,
          "avec ε = 0,3, compare encore les deux termes de chaque transition : le terme clippé est-il quelque part strictement le plus petit ?": 0,
          "relis les quatre ratios contre [0,7 ; 1,3]": 3})

WB_ANSWER {"decimals": 2, "hash": "f16e79cf37ad5156447e9a8c8a0ef58a02747aedfdf10001ca3d36c8e5623c2c", "hash_coarse": "7cf08f369f360cce552c3c6e5243804eeccc51fdf6bbdad3a45a753fa3e6366e", "id": "B8.5a", "kind": "float", "magnitude": 0, "mistakes": {"a6bd8f9f0f9849f7d78e7f440924321f4506608cb47e664c0edd58c7294894bb": "le clip porte sur le ratio, pas sur le produit", "d1e05531bf4623a8e5035b3bedae3c2f9ff50f3d201e1ca51a78cb1720ed2d23": "le min prend le plus petit des deux termes : relis le cas de l'avantage positif (§208.5)"}, "sign": 1}
📝 Ex B8.5a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "9eff3385217f072f6555dd10a8eb3f2e3957fada1c069709103a62cc004bb6e0", "hash_coarse": "401ba529191f6b0692bc2cba28e7db06d3fe81668be29c5a976dacf20fe5b428", "id": "B8.5b", "kind": "float", "magnitude": -1, "mistakes": {"287b69c80b811a4622c15efa8d7e7b4b6c5cd770822244a5383ceca69eb6988b": "le min prend le plus petit des deux termes : avec un avantage négatif, lequel est le plus p

**Ex B8.6 — Préférences : loss de Bradley-Terry puis loss DPO sur une paire de réponses**

In [23]:
wb.record("B8.6a", P_B86, decimals=3, mistakes={"vérifie le sens de la différence : la préférée moins l'autre": sigmoid(RL_B86 - RW_B86),
          "Bradley-Terry passe la différence par une sigmoïde": RW_B86 / (RW_B86 + RL_B86)})
wb.record("B8.6b", LOSS_B86, decimals=3, mistakes={"la loss est −log σ(d), le logarithme naturel": -math.log10(P_B86),
          "vérifie le sens de la différence : la préférée moins l'autre": LOSS_SWAP_B86,
          "relis la question : la loss, pas la probabilité": 1 - P_B86})
wb.record("B8.6c", GRAD_W_B86, decimals=3, mistakes={"vérifie le signe : monter r(y_w) fait baisser la loss": -GRAD_W_B86,
          "la dérivée de −log σ(d) vaut −(1 − σ(d)), pas −σ(d)": -P_B86})
wb.record("B8.6d", LOSS_SWAP_B86, decimals=3, mistakes={"relis la question : le choix inverse, avec les mêmes récompenses": LOSS_B86,
          "la loss est −log σ(d), le logarithme naturel": -math.log10(1 - P_B86)})
wb.record("B8.6e", True, mistakes={"relis « Le modèle de récompense de Bradley-Terry » (§208.7) : de quoi dépend la loss ?": False})
wb.record("B8.6f", IMPLICIT_B86, decimals=2, mistakes={"vérifie le sens du logarithme : la politique sur la référence": -IMPLICIT_B86,
          "relis la question : multiplie par β": IMPLICIT_B86 / 0.1,
          "relis la question : la récompense implicite de y_w seule": MARGIN_B86})
wb.record("B8.6g", MARGIN_B86, decimals=2, mistakes={"relis la formule de DPO (§208.7) : β multiplie la différence": MARGIN_B86 / 0.1,
          "vérifie le signe de la partie de y_l : on retranche son log-ratio": 0.1 * ((LOGP_B86["pc"] - LOGP_B86["rc"]) + (LOGP_B86["pr"] - LOGP_B86["rr"])),
          "vérifie le sens des log-ratios : la politique moins la référence": -MARGIN_B86})
wb.record("B8.6h", DPO_B86, decimals=3, mistakes={"vérifie le signe de la marge dans −log σ(·)": softplus(MARGIN_B86),
          "relis la formule de DPO : β multiplie la différence": softplus(-MARGIN_B86 / 0.1),
          "la loss est −log σ(marge), le logarithme naturel": -math.log10(sigmoid(MARGIN_B86))})
wb.record("B8.6i", DPO5_B86, decimals=3, mistakes={"relis la question : β = 0,5": DPO_B86,
          "vérifie le signe de la marge dans −log σ(·)": softplus(5 * MARGIN_B86)})
wb.record("B8.6j", DPO_GRAD_B86, decimals=4, mistakes={"vérifie le signe : monter log π(y_w) fait baisser la loss": -DPO_GRAD_B86,
          "la dérivée de la marge par rapport à log π(y_w) vaut β : n'oublie pas ce facteur": DPO_GRAD_B86 / 0.1,
          "la dérivée de −log σ(m) vaut −σ(−m), pas −σ(m)": -0.1 * sigmoid(MARGIN_B86)})

WB_ANSWER {"decimals": 3, "hash": "5f0a4ef8d963156f2209d70c90d01d2783e94cfbfc32fdb637aaea93f3981c19", "hash_coarse": "c747a1f19c039d57d1063cc4c7758f9c49baf01a92d2f4d4e6c55df8d6db63c2", "id": "B8.6a", "kind": "float", "magnitude": -1, "mistakes": {"25b11d155b145d36257e2d20d31f619594f0d18725a0d23e663b59fd4c00ebf2": "vérifie le sens de la différence : la préférée moins l'autre", "53249d4f1bad9782d8ccd5fb371edadb12c206c2e8f318d754abbca39d0f44b9": "Bradley-Terry passe la différence par une sigmoïde"}, "sign": 1}
📝 Ex B8.6a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "1e374a23f0289ea000accdd1f2e619dd0ee1ef07b24e49ddc7fbac0ece948e97", "hash_coarse": "1521f53fbd0e54aaa0e7f8b68cd335e97274bc141b24e22b0e439491838f7dd6", "id": "B8.6b", "kind": "float", "magnitude": -1, "mistakes": {"4e6cb7e4fafebd72e128ff6769de3312844b8facb501b7896ecd36bd7ff5806f": "la loss est −log σ(d), le logarithme naturel", "ac2787295a1c6338e5f7abdcebff7471e0fdf30ebce767fd8d605d17f3d2d5e8":

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une cible DQN et Double DQN, en distinguant une vraie fin d'une limite de temps ?
2. Sais-tu dériver le gradient du log d'une politique softmax, et dire pourquoi une baseline ne biaise pas le gradient ?
3. Sais-tu calculer GAE à rebours et l'objectif clippé de PPO avec son gradient, cas par cas ?
4. Sais-tu calculer une loss de Bradley-Terry et une loss DPO, et dire ce que règle $\beta$ ?

**Pour aller plus loin** : l'article de Schulman et coll. (📄 B8.11) et les « 37 détails » de Huang et coll., cités dans la fiche. B8 est le dernier chapitre bonus.

*Les exercices de code B8.12 à B8.28 (la valeur, la politique, les préférences) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B8` pour obtenir le notebook complet.*